# 01 — Load Titanic CSV into a Spark DataFrame

**Output table:** `ctl_training_dev.sudsuay_titanic` (bronze / raw layer)

This notebook does one job: get the Kaggle CSV into Spark faithfully and persist it as a
managed table. No cleaning, no feature engineering — that belongs in notebook 02.

Two things about this particular CSV need care:

1. The header contains the name `zero` **19 times**. Spark refuses to build a schema with
   duplicate column names, so we de-duplicate the header ourselves before reading.
2. The label column is named `2urvived`. A leading digit makes it awkward to reference in
   Spark SQL, so it is renamed to `Survived` at load time. Every rename is printed below.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("titanic_load")
    .getOrCreate()
)

# On Databricks this returns the session that already exists; locally it creates one.
print("Spark version:", spark.version)

## 1. Configuration

`CSV_PATH` is read with plain Python (not Spark) to peek at the header, so it must be a
path on the **driver's local filesystem**. On Databricks that means a workspace file,
`/dbfs/...`, or `/Volumes/...` — note the `/dbfs` prefix rather than `dbfs:/`.

In [ ]:
CSV_PATH   = "train_and_test2.csv"          # <-- point at your local copy of the Kaggle file
CATALOG    = None                            # e.g. "main"; leave None for a 2-level name
SCHEMA     = "ctl_training_dev"
TABLE      = "sudsuay_titanic"
TABLE_FMT  = "delta"                         # falls back to parquet if Delta is unavailable

TABLE_FQN = ".".join(x for x in [CATALOG, SCHEMA, TABLE] if x)
print("Target table:", TABLE_FQN)

## 2. Build a schema with de-duplicated column names

Rather than letting Spark infer the schema (which fails on the duplicate `zero` columns),
we read the header line, make the names unique, and hand Spark an explicit schema.
Passing an explicit schema alongside `header=True` makes Spark skip the header row and
use our names instead.

In [ ]:
import re

DOUBLE_COLS = {"age", "fare", "embarked"}   # the only genuinely continuous / nullable fields

RENAMES = {
    "passengerid": "PassengerId",
    "sibsp":       "SibSp",
    "parch":       "Parch",
    "2urvived":    "Survived",   # leading digit -> unusable without backticks in Spark SQL
}


def dedupe_header(names):
    """Make column names unique and Spark-SQL friendly. Returns (clean_names, rename_log)."""
    seen, out, log = {}, [], []
    zero_n = 0
    for raw in names:
        key = raw.strip().lower()
        if key == "zero":
            zero_n += 1
            new = f"zero_{zero_n:02d}"
        elif key in RENAMES:
            new = RENAMES[key]
        else:
            new = re.sub(r"[^0-9a-zA-Z_]", "_", raw.strip())
        # guard against any remaining collision
        if new in seen:
            seen[new] += 1
            new = f"{new}_{seen[new]}"
        else:
            seen[new] = 0
        if new != raw.strip():
            log.append((raw.strip(), new))
        out.append(new)
    return out, log


with open(CSV_PATH) as fh:
    raw_header = fh.readline().strip().split(",")

columns, rename_log = dedupe_header(raw_header)

schema = StructType([
    StructField(c, DoubleType() if c.lower() in DOUBLE_COLS else IntegerType(), True)
    for c in columns
])

print(f"{len(raw_header)} raw columns -> {len(set(columns))} unique names\n")
print("Renames applied:")
for old, new in rename_log:
    print(f"  {old!r:<16} -> {new}")

## 3. Read the CSV into a Spark DataFrame

In [ ]:
df = (
    spark.read
    .option("header", True)          # skip the header row; use our schema's names
    .option("mode", "PERMISSIVE")
    .option("nullValue", "")
    .schema(schema)
    .csv(CSV_PATH)
)

df = df.withColumn("_ingested_at", F.current_timestamp())

df.printSchema()

In [ ]:
df.limit(5).toPandas()

## 4. Sanity checks

Just enough to prove the load worked. Real profiling happens in notebook 02.

In [ ]:
n_rows = df.count()
n_ids  = df.select("PassengerId").distinct().count()

print(f"rows                : {n_rows:,}")
print(f"distinct PassengerId: {n_ids:,}")
print(f"columns             : {len(df.columns)}")
assert n_rows == n_ids, "PassengerId is not unique — investigate before writing the table"

null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
]).toPandas().T.rename(columns={0: "nulls"})

print("\nColumns containing nulls:")
print(null_counts[null_counts["nulls"] > 0].to_string() or "  (none)")

In [ ]:
# Label balance as loaded — carried forward as-is; notebook 02 explains the caveat.
df.groupBy("Survived").count().orderBy("Survived").show()

## 5. Write to `ctl_training_dev.sudsuay_titanic`

`overwrite` + `overwriteSchema` makes the notebook safely re-runnable.

In [ ]:
schema_fqn = ".".join(x for x in [CATALOG, SCHEMA] if x)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_fqn}")


def save_table(sdf, fqn, fmt="delta"):
    """Write a managed table, degrading to parquet where Delta isn't available."""
    try:
        (sdf.write.format(fmt)
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(fqn))
        print(f"wrote {fqn} as {fmt}")
    except Exception as exc:
        reason = str(exc).strip().splitlines()[0][:160]
        print(f"[warn] {fmt} write failed ({type(exc).__name__}: {reason}); retrying as parquet")
        (sdf.write.format("parquet")
            .mode("overwrite")
            .saveAsTable(fqn))
        print(f"wrote {fqn} as parquet")


save_table(df, TABLE_FQN, TABLE_FMT)

## 6. Verify the round-trip

In [ ]:
check = spark.table(TABLE_FQN)
print(f"{TABLE_FQN}: {check.count():,} rows x {len(check.columns)} columns")
check.limit(5).toPandas()

---

**Done.** `ctl_training_dev.sudsuay_titanic` holds the raw dataset, 1,309 rows, with unique
column names and nothing else changed.

Next: **02 — data preparation & EDA** → `ctl_training_dev.sudsuay_titanic_prep`